In [1]:
import numpy as np
import pandas as pd
import openmatrix as omx
from pathlib import Path
from dbfread import DBF
import copy
from datetime import datetime
import os
import geopandas as gpd
from libpysal.weights import KNN


In [2]:
# import global TDM functions
import sys

sys.path.insert(0, "../Resources/2-Python/global-functions")
import BigQuery

client = BigQuery.getBigQueryClient_Confidential2023UtahHTS()


In [ ]:
path_inputs = Path.cwd() / "inputs"
path_start_files = path_inputs / "input_files/C28"
path_taz_file = path_start_files / "WFv1000_TAZ.dbf"

df_taz = pd.DataFrame(iter(DBF(path_taz_file)))
df_taz = df_taz[["TAZID", "TAZID_V911", "CO_TAZID", "DISTLRG"]]

omx_hbw_folder = (
    r"../WF-TDM-Documentation/_large_files/v1000/v1000-C28/2b-hbwdestchoice/"
)


In [3]:
hts_hh_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.hh"
).to_dataframe()
hts_person_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.person"
).to_dataframe()
hts_trip_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.trip_linked"
).to_dataframe()
hts_veh_23 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2023.vehicle"
).to_dataframe()


c:\Users\cday\Anaconda3\envs\base0426\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
hts_hh_12 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2012.household"
).to_dataframe()
hts_person_12 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2012.person"
).to_dataframe()
hts_trip_12 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2012.trip"
).to_dataframe()
hts_veh_12 = client.query(
    "SELECT * FROM " + "wfrc-modeling-data.prd_tdm_hts_2012.vehicle"
).to_dataframe()


### Observed 2023


In [67]:
hts_trip_23_merge = hts_trip_23.copy()
hts_trip_23_merge = hts_trip_23_merge[
    [
        "unique_id",
        "hh_id",
        "person_id",
        "vehicle_id",
        "pCO_TAZID_USTMv4",
        "aCO_TAZID_USTMv4",
        "pSUBAREAID",
        "aSUBAREAID",
        "PURP7_t",
        "trip_weight",
        "distance_miles",
    ]
]

# filter to HBW
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["PURP7_t"] == "HBW"]

# merge taz
hts_trip_23_merge = hts_trip_23_merge.merge(
    df_taz[["TAZID", "CO_TAZID"]],
    how="left",
    left_on="pCO_TAZID_USTMv4",
    right_on="CO_TAZID",
)
hts_trip_23_merge = hts_trip_23_merge.drop(columns="CO_TAZID").rename(
    columns={"TAZID": "pTAZID"}
)
hts_trip_23_merge = hts_trip_23_merge.merge(
    df_taz[["TAZID", "CO_TAZID"]],
    how="left",
    left_on="aCO_TAZID_USTMv4",
    right_on="CO_TAZID",
)
hts_trip_23_merge = hts_trip_23_merge.drop(columns="CO_TAZID").rename(
    columns={"TAZID": "aTAZID"}
)

# fitler to WF subarea
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["pSUBAREAID"] == 1]
hts_trip_23_merge = hts_trip_23_merge[hts_trip_23_merge["aSUBAREAID"] == 1]

# vehicle ownership lookup table
vehown_lookup = (
    hts_veh_23.copy()
    .groupby("hh_id")["vehicle_id"]
    .count()
    .reset_index(name="veh_count")
)
vehown_lookup["vehown"] = vehown_lookup["veh_count"].clip(upper=2)
vehown_lookup = vehown_lookup[["hh_id", "vehown"]]

# income lookup table
income_lookup = hts_hh_23.copy()[["hh_id", "income_detailed"]]
income_lookup["income"] = np.select(
    [
        income_lookup["income_detailed"].isin([1, 2, 3, 4]),
        income_lookup["income_detailed"].isin([5, 6, 7, 8, 9, 10]),
    ],
    ["lo", "hi"],
    default="unknown",
)

income_lookup = income_lookup[["hh_id", "income"]]

# 1. Merge vehicle lookup
hts_trip_23_merge = hts_trip_23_merge.merge(vehown_lookup, how="left", on="hh_id")

# 2. Re-establish 0 vehicles for missing households (and strip decimals)
hts_trip_23_merge["vehown"] = hts_trip_23_merge["vehown"].fillna(0).astype("Int64")

# 3. Merge income lookup
hts_trip_23_merge = hts_trip_23_merge.merge(income_lookup, how="left", on="hh_id")

# 4. Fill missing income with "unknown"
hts_trip_23_merge["income"] = hts_trip_23_merge["income"].fillna("unknown")

# 5. Calculate segment
# Since vehown is now always a number (0, 1, or 2),
# we only label 'unknown' if the income data is missing.
hts_trip_23_merge["segment"] = np.where(
    (hts_trip_23_merge["income"] == "unknown") | (hts_trip_23_merge["income"] == ""),
    "unknown",
    hts_trip_23_merge["vehown"].astype(str)
    + "veh_"
    + hts_trip_23_merge["income"].astype(str),
)

# 6. Final table selection
hts_trip_23_final = hts_trip_23_merge[
    ["segment", "vehown", "income", "pTAZID", "aTAZID", "trip_weight"]
]

hts_trip_23_final


,segment,vehown,income,pTAZID,aTAZID,trip_weight
0,1veh_lo,1,lo,2979.0,2966.0,17.687357
1,2veh_hi,2,hi,2779.0,2983.0,53.679940
2,2veh_hi,2,hi,830.0,866.0,101.865387
3,unknown,2,unknown,3058.0,3041.0,0.000000
4,2veh_hi,2,hi,1909.0,1971.0,292.121640
...,...,...,...,...,...,...
14972,1veh_hi,1,hi,512.0,958.0,22.263372
14973,2veh_hi,2,hi,1792.0,1786.0,456.865792
14974,2veh_hi,2,hi,1684.0,1467.0,0.000000
14975,0veh_lo,0,lo,1124.0,1007.0,0.000000


### Observed 2012


In [66]:
hts_trip_12_merge = hts_trip_12.copy()
hts_trip_12_merge = hts_trip_12_merge[
    [
        "record_id",
        "hptripid",
        "password",
        "hhmemberid",
        "p_SA1_TAZID",
        "a_SA1_TAZID",
        "p_SUBAREA_v30",
        "a_SUBAREA_v30",
        "PURP7_t",
        "weight",
    ]
]


# filter to HBW
hts_trip_12_merge = hts_trip_12_merge[hts_trip_12_merge["PURP7_t"] == "HBW"]

# merge taz
hts_trip_12_merge = hts_trip_12_merge.merge(
    df_taz[["TAZID_V911", "TAZID"]],
    how="left",
    left_on="p_SA1_TAZID",
    right_on="TAZID_V911",
)
hts_trip_12_merge = hts_trip_12_merge.drop(
    columns=["p_SA1_TAZID", "TAZID_V911"]
).rename(columns={"TAZID": "pTAZID"})
hts_trip_12_merge = hts_trip_12_merge.merge(
    df_taz[["TAZID_V911", "TAZID"]],
    how="left",
    left_on="a_SA1_TAZID",
    right_on="TAZID_V911",
)
hts_trip_12_merge = hts_trip_12_merge.drop(
    columns=["a_SA1_TAZID", "TAZID_V911"]
).rename(columns={"TAZID": "aTAZID"})

# fitler to WF subarea
hts_trip_12_merge = hts_trip_12_merge[hts_trip_12_merge["p_SUBAREA_v30"] == 1]
hts_trip_12_merge = hts_trip_12_merge[hts_trip_12_merge["a_SUBAREA_v30"] == 1]


# vehicle ownership lookup table
vehown_lookup = (
    hts_veh_12.copy().groupby("password").size().reset_index(name="veh_count")
)
vehown_lookup["vehown"] = vehown_lookup["veh_count"].clip(upper=2)
vehown_lookup = vehown_lookup[["password", "vehown"]]

# income lookup table
income_lookup = hts_hh_12.copy()[["password", "hh_income_cat"]]
income_lookup["income"] = np.select(
    [
        income_lookup["hh_income_cat"].isin([1, 2]),
        income_lookup["hh_income_cat"].isin([3, 4, 5]),
    ],
    ["lo", "hi"],
    default="unknown",  # <-- Catch unclassified income instead of using ""
)
income_lookup = income_lookup[["password", "income"]]

# merge vehown and income to trip table
hts_trip_12_merge = hts_trip_12_merge.merge(vehown_lookup, how="left", on="password")

hts_trip_12_merge["vehown"] = hts_trip_12_merge["vehown"].fillna(0).astype("Int64")

hts_trip_12_merge = hts_trip_12_merge.merge(income_lookup, how="left", on="password")

hts_trip_12_merge["income"] = hts_trip_12_merge["income"].fillna("unknown")

# 3. Corrected Segment Logic
hts_trip_12_merge["segment"] = np.where(
    (hts_trip_12_merge["income"] == "unknown") | (hts_trip_12_merge["income"] == ""),
    "unknown",  # Only unknown if income is missing
    hts_trip_12_merge["vehown"].astype(str)
    + "veh_"
    + hts_trip_12_merge["income"].astype(str),
)

hts_trip_12_final = hts_trip_12_merge[
    ["segment", "vehown", "income", "pTAZID", "aTAZID", "weight"]
]
hts_trip_12_final


,segment,vehown,income,pTAZID,aTAZID,weight
0,2veh_hi,2,hi,150.0,629.0,48.377805
1,2veh_hi,2,hi,29.0,977.0,63.424559
2,2veh_hi,2,hi,40.0,235.0,62.759109
3,2veh_hi,2,hi,40.0,235.0,62.759109
4,unknown,2,unknown,33.0,235.0,53.518451
...,...,...,...,...,...,...
10818,2veh_lo,2,lo,432.0,833.0,146.568901
10819,2veh_lo,2,lo,535.0,885.0,188.100313
10820,2veh_lo,2,lo,535.0,885.0,188.100313
10821,unknown,2,unknown,566.0,890.0,263.357977


### Merge Observed


In [68]:
# The keys we want to match on
merge_keys = ["segment", "vehown", "income", "pTAZID", "aTAZID"]

# 1. Aggregate 2012 trips down to unique combinations and sum the weights
hts_12_agg = (
    hts_trip_12_final.groupby(merge_keys, dropna=False)["weight"]
    .sum()
    .reset_index()
    .rename(columns={"weight": "trip_weight_12"})
)

# 2. Aggregate 2023 trips down to unique combinations and sum the weights
hts_23_agg = (
    hts_trip_23_final.groupby(merge_keys, dropna=False)["trip_weight"]
    .sum()
    .reset_index()
    .rename(columns={"trip_weight": "trip_weight_23"})
)

# 3. Outer merge the aggregated data
obs_merge = hts_12_agg.merge(
    hts_23_agg,
    how="outer",
    on=merge_keys,
)

# 4. Merge Origin District
obs_merge = obs_merge.merge(
    df_taz[["TAZID", "DISTLRG"]], how="left", left_on="pTAZID", right_on="TAZID"
)
obs_merge = obs_merge.rename(columns={"DISTLRG": "p_DISTLRG"}).drop(columns="TAZID")

# 5. Merge Destination District
obs_merge = obs_merge.merge(
    df_taz[["TAZID", "DISTLRG"]], how="left", left_on="aTAZID", right_on="TAZID"
)
obs_merge = obs_merge.rename(columns={"DISTLRG": "a_DISTLRG"}).drop(columns="TAZID")

# 6. Fill NaNs (Missing weights become 0)
obs_merge = obs_merge.fillna(0)

# 7. Final Column Ordering
obs_merge = obs_merge[
    [
        "segment",
        "vehown",
        "income",
        "pTAZID",
        "aTAZID",
        "p_DISTLRG",
        "a_DISTLRG",
        "trip_weight_12",
        "trip_weight_23",
    ]
]
obs_merge


,segment,vehown,income,pTAZID,aTAZID,p_DISTLRG,a_DISTLRG,trip_weight_12,trip_weight_23
0,0veh_hi,0,hi,377.0,426.0,5.0,5.0,187.790285,0.000000
1,0veh_hi,0,hi,440.0,437.0,5.0,5.0,214.664782,0.000000
2,0veh_hi,0,hi,883.0,1109.0,9.0,12.0,0.000000,326.911234
3,0veh_hi,0,hi,884.0,836.0,9.0,9.0,0.000000,64.448140
4,0veh_hi,0,hi,884.0,1536.0,9.0,14.0,0.000000,193.344420
...,...,...,...,...,...,...,...,...,...
11531,unknown,2,unknown,3432.0,2771.0,22.0,20.0,0.000000,745.705223
11532,unknown,2,unknown,3443.0,2646.0,22.0,20.0,155.057298,0.000000
11533,unknown,2,unknown,3443.0,3027.0,22.0,21.0,232.585947,0.000000
11534,unknown,2,unknown,3467.0,3497.0,22.0,22.0,147.614135,0.000000


In [69]:
hts_trip_12_hbw = hts_trip_12.copy()
hts_trip_12_hbw = hts_trip_12_hbw[hts_trip_12_hbw["PURP7_t"] == "HBW"]
hts_trip_12_hbw = hts_trip_12_hbw[hts_trip_12_hbw["p_SUBAREA_v30"] == 1]
hts_trip_12_hbw = hts_trip_12_hbw[hts_trip_12_hbw["a_SUBAREA_v30"] == 1]
print(hts_trip_12_hbw["weight"].sum())
print(obs_merge["trip_weight_12"].sum())

hts_trip_23_hbw = hts_trip_23.copy()
hts_trip_23_hbw = hts_trip_23_hbw[hts_trip_23_hbw["PURP7_t"] == "HBW"]
hts_trip_23_hbw = hts_trip_23_hbw[hts_trip_23_hbw["pSUBAREAID"] == 1]
hts_trip_23_hbw = hts_trip_23_hbw[hts_trip_23_hbw["aSUBAREAID"] == 1]
print(hts_trip_23_hbw["trip_weight"].sum())
print(obs_merge["trip_weight_23"].sum())


1228269.88801182
1228817.03028614
1711202.513165094
1711202.513165094


In [70]:
obs_merge.to_csv("observed_data_12_23.csv", index=False)


### Census Data x LEHD Data


In [ ]:
import requests
import pandas as pd
import gzip
import zipfile
from io import BytesIO


def get_acs_zero_vehicle_pct():
    print("1. Fetching ACS demographics (0-Vehicle Households)...")
    url = "https://api.census.gov/data/2023/acs/acs5?get=B08141_001E,B08141_002E&for=tract:*&in=state:49"

    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"API Failed: {response.text}")

    data = response.json()
    df_acs = pd.DataFrame(data[1:], columns=data[0])

    df_acs["Total_Workers"] = pd.to_numeric(
        df_acs["B08141_001E"], errors="coerce"
    ).fillna(0)
    df_acs["Zero_Vehicles"] = pd.to_numeric(
        df_acs["B08141_002E"], errors="coerce"
    ).fillna(0)

    df_acs["pct_zero_veh"] = 0.0
    valid_workers = df_acs["Total_Workers"] > 0
    df_acs.loc[valid_workers, "pct_zero_veh"] = (
        df_acs["Zero_Vehicles"] / df_acs["Total_Workers"]
    )

    # 11-digit Tract ID
    df_acs["tract_id"] = df_acs["state"] + df_acs["county"] + df_acs["tract"]
    return df_acs[["tract_id", "Total_Workers", "Zero_Vehicles", "pct_zero_veh"]]


def get_lodes_od_flows():
    print("2. Downloading LODES origin-destination flows (this may take a minute)...")
    url = "https://lehd.ces.census.gov/data/lodes/LODES8/ut/od/ut_od_main_JT00_2023.csv.gz"

    response = requests.get(url)
    with gzip.open(BytesIO(response.content), "rt") as f:
        df_lodes = pd.read_csv(f)

    df_lodes["home_tract"] = df_lodes["h_geocode"].astype(str).str.zfill(15).str[:11]
    df_lodes["work_tract"] = df_lodes["w_geocode"].astype(str).str.zfill(15).str[:11]

    target_counties = ["49003", "49011", "49035", "49049", "49057"]
    df_region = df_lodes[df_lodes["home_tract"].str[:5].isin(target_counties)]

    tract_flows = (
        df_region.groupby(["home_tract", "work_tract"])["S000"].sum().reset_index()
    )
    tract_flows.rename(columns={"S000": "total_flow"}, inplace=True)
    return tract_flows


def get_tract_centroids():
    print("3. Fetching Census Tract centroids (Lat/Lon)...")
    # Pulls the official Census Gazetteer file for tract geometries
    url = "https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2023_Gazetteer/2023_Gaz_tracts_national.zip"

    response = requests.get(url)
    with zipfile.ZipFile(BytesIO(response.content)) as z:
        # Find the .txt file inside the zip archive
        txt_filename = [name for name in z.namelist() if name.endswith(".txt")][0]
        with z.open(txt_filename) as f:
            # Gazetteer files are tab-separated
            df_gaz = pd.read_csv(f, sep="\t", dtype={"GEOID": str})

    # Clean column names (Census text files often have trailing spaces)
    df_gaz.columns = df_gaz.columns.str.strip()

    # Isolate just the GEOID and the Latitude/Longitude
    df_centroids = df_gaz[["GEOID", "INTPTLAT", "INTPTLONG"]].copy()
    df_centroids.rename(columns={"INTPTLAT": "lat", "INTPTLONG": "lon"}, inplace=True)

    return df_centroids


def build_estimated_od_matrix():
    df_acs = get_acs_zero_vehicle_pct()
    df_flows = get_lodes_od_flows()
    df_centroids = get_tract_centroids()

    print("4. Merging datasets and estimating 0-vehicle flows...")

    # Merge Flow data with Origin (Home) Demographics
    df_merged = pd.merge(
        df_flows, df_acs, left_on="home_tract", right_on="tract_id", how="inner"
    )

    # Calculate the Estimated 0-Vehicle Flow
    df_merged["est_0_veh_flow"] = (
        (df_merged["total_flow"] * df_merged["pct_zero_veh"]).round().astype(int)
    )

    print("5. Attaching Latitude & Longitude to Home and Work tracts...")
    # Merge Centroids onto the Home Tract
    df_merged = pd.merge(
        df_merged, df_centroids, left_on="home_tract", right_on="GEOID", how="left"
    )
    df_merged.rename(columns={"lat": "home_lat", "lon": "home_lon"}, inplace=True)
    df_merged.drop(columns=["GEOID"], inplace=True)

    # Merge Centroids onto the Work Tract
    df_merged = pd.merge(
        df_merged, df_centroids, left_on="work_tract", right_on="GEOID", how="left"
    )
    df_merged.rename(columns={"lat": "work_lat", "lon": "work_lon"}, inplace=True)
    df_merged.drop(columns=["GEOID"], inplace=True)

    # Sort and reorder columns for a clean output matrix
    df_final = df_merged.sort_values(by="est_0_veh_flow", ascending=False)
    output_cols = [
        "home_tract",
        "home_lat",
        "home_lon",
        "work_tract",
        "work_lat",
        "work_lon",
        "total_flow",
        "pct_zero_veh",
        "est_0_veh_flow",
    ]
    df_final = df_final[output_cols]

    # Save and return
    output_filename = "wasatch_front_0veh_flows_with_centroids.csv"
    df_final.to_csv(output_filename, index=False)

    print(f"\nSuccess! Processed {len(df_final):,} Origin-Destination pairs.")
    print(f"Data saved locally as: {output_filename}")

    return df_final


# Run the master function
if __name__ == "__main__":
    od_est_mtx = build_estimated_od_matrix()
    display(od_est_mtx.head())


1. Fetching ACS demographics (0-Vehicle Households)...
2. Downloading LODES origin-destination flows (this may take a minute)...
3. Fetching Census Tract centroids (Lat/Lon)...
4. Merging datasets and estimating 0-vehicle flows...
5. Attaching Latitude & Longitude to Home and Work tracts...

Success! Processed 187,390 Origin-Destination pairs.
Data saved locally as: wasatch_front_0veh_flows_with_centroids.csv


,home_tract,home_lat,home_lon,work_tract,work_lat,work_lon,total_flow,pct_zero_veh,est_0_veh_flow
106945,49035113904,40.700099,-112.077431,49035114500,40.734878,-111.980084,396,0.136906,54
33549,49035101900,40.765004,-111.875349,49035101402,40.767320,-111.827564,222,0.228881,51
33689,49035101900,40.765004,-111.875349,49035114000,40.757595,-111.897457,194,0.228881,44
32401,49035101500,40.765007,-111.857402,49035101402,40.767320,-111.827564,266,0.129327,34
56652,49035112101,40.668099,-111.894952,49035114500,40.734878,-111.980084,190,0.151186,29


In [ ]:
# conver to large district level
import pandas as pd
import geopandas as gpd


def aggregate_to_large_districts(od_df, shapefile_path):
    print(f"1. Loading TAZ shapefile from: {shapefile_path}")
    # Read the shapefile
    taz_gdf = gpd.read_file(shapefile_path)

    # CRITICAL: Shapefiles from WFRC/MAG are usually in a local state plane projection (e.g., NAD83).
    # We must convert the shapefile to WGS84 (EPSG:4326) so it matches the Census Latitude/Longitude.
    print("2. Reprojecting shapefile to match Census coordinates (WGS84)...")
    taz_gdf = taz_gdf.to_crs(epsg=4326)

    # Keep only the geometry and the DISTLRG column to save memory
    taz_gdf = taz_gdf[["DISTLRG", "geometry"]]

    print("3. Performing spatial join for HOME locations...")
    # Convert the home lat/lon coordinates into geographic Point objects
    home_gdf = gpd.GeoDataFrame(
        od_df,
        geometry=gpd.points_from_xy(od_df.home_lon, od_df.home_lat),
        crs="EPSG:4326",
    )
    # Point-in-Polygon spatial join
    home_joined = gpd.sjoin(home_gdf, taz_gdf, how="left", predicate="intersects")

    # Drop any duplicates just in case a point landed exactly on a boundary line
    home_joined = home_joined[~home_joined.index.duplicated(keep="first")]

    print("4. Performing spatial join for WORK locations...")
    # Convert the work lat/lon coordinates into geographic Point objects
    work_gdf = gpd.GeoDataFrame(
        od_df,
        geometry=gpd.points_from_xy(od_df.work_lon, od_df.work_lat),
        crs="EPSG:4326",
    )
    # Point-in-Polygon spatial join
    work_joined = gpd.sjoin(work_gdf, taz_gdf, how="left", predicate="intersects")
    work_joined = work_joined[~work_joined.index.duplicated(keep="first")]

    print("5. Aggregating flows to the DISTLRG level...")
    # Assign the new District IDs back to the original dataframe
    od_df["home_DISTLRG"] = home_joined["DISTLRG"]
    od_df["work_DISTLRG"] = work_joined["DISTLRG"]

    # Optional: Fill any missing districts with -1 (e.g., if a tract centroid fell slightly outside the shapefile)
    od_df["home_DISTLRG"] = od_df["home_DISTLRG"].fillna(-1).astype(int)
    od_df["work_DISTLRG"] = od_df["work_DISTLRG"].fillna(-1).astype(int)

    # Group by the new Large Districts and sum the flows
    district_od = (
        od_df.groupby(["home_DISTLRG", "work_DISTLRG"])[
            ["total_flow", "est_0_veh_flow"]
        ]
        .sum()
        .reset_index()
    )

    # Sort to show the highest volume district-to-district flows first
    district_od = district_od.sort_values(by="est_0_veh_flow", ascending=False)

    # Save the final matrix
    output_filename = "wasatch_front_DISTLRG_0veh_flows.csv"
    district_od.to_csv(output_filename, index=False)
    print(f"\nSuccess! District-level OD matrix saved as: {output_filename}")

    return district_od


# --- Execute the code ---
# Ensure you pass the od_est_mtx variable we created in the previous step
shapefile_location = "inputs/input_files/C28/WFv1000_taz.shp"

# Run the function
district_od_matrix = aggregate_to_large_districts(od_est_mtx, shapefile_location)

# Display the final District-to-District results
display(district_od_matrix)


1. Loading TAZ shapefile from: inputs/input_files/C28/WFv1000_taz.shp
2. Reprojecting shapefile to match Census coordinates (WGS84)...
3. Performing spatial join for HOME locations...
4. Performing spatial join for WORK locations...
5. Aggregating flows to the DISTLRG level...

Success! District-level OD matrix saved as: wasatch_front_DISTLRG_0veh_flows.csv


,home_DISTLRG,work_DISTLRG,total_flow,est_0_veh_flow
377,14,14,50907,1137
324,12,12,42564,696
433,16,16,36858,688
375,14,12,25077,609
570,21,21,35415,603
...,...,...,...,...
10,-1,10,45,0
671,25,19,70,0
670,25,18,61,0
3,-1,3,66,0


### Modeled Data


In [90]:
# --------------------------------------------------------------------#
# Process Modeled Data (OMX Files)                                   #
# --------------------------------------------------------------------#
folder = "omx_hbw_folder"
files = [
    "pa_HBW_0veh_hi.omx",
    "pa_HBW_0veh_lo.omx",
    "pa_HBW_1veh_hi.omx",
    "pa_HBW_1veh_lo.omx",
    "pa_HBW_2veh_hi_noXI.omx",
    "pa_HBW_2veh_lo_noXI.omx",
]


def parse_veh_inc(fname):
    base = os.path.splitext(fname)[0]  # pa_HBW_2veh_hi_noXI
    base = base.replace("pa_", "")  # HBW_2veh_hi_noXI
    base = base.replace("_noXI", "")  # HBW_2veh_hi
    return base


dfs = []

for f in files:
    with omx.open_file(os.path.join(folder, f), "r") as omx_file:
        mat = omx_file["trips"][:]

    df = (
        pd.DataFrame(mat)
        .reset_index()
        .melt(id_vars="index", var_name="a_taz", value_name="trips")
        .rename(columns={"index": "p_taz"})
    )

    # shift from 0-based to 1-based TAZ
    df["p_taz"] += 1
    df["a_taz"] += 1

    df["veh_inc"] = parse_veh_inc(f)

    dfs.append(df)

mod_taz_hbw_0 = pd.concat(dfs, ignore_index=True)

# --------------------------------------------------------------------#
# Summarize Modeled Data                                             #
# --------------------------------------------------------------------#
# merge to get large districts and summarize accordingly
mod_taz_hbw_1 = mod_taz_hbw_0.merge(
    tazv10_wf, how="left", left_on="p_taz", right_on="TAZID"
)
mod_taz_hbw_1 = mod_taz_hbw_1.rename(columns={"DISTLRG": "p_DistLrg"}).drop(
    columns={"TAZID"}
)
mod_taz_hbw_2 = mod_taz_hbw_1.merge(
    tazv10_wf, how="left", left_on="a_taz", right_on="TAZID"
)
mod_taz_hbw_2 = mod_taz_hbw_2.rename(columns={"DISTLRG": "a_DistLrg"}).drop(
    columns={"TAZID"}
)

mod_dist_hbw_sum = (
    mod_taz_hbw_2[["veh_inc", "p_DistLrg", "a_DistLrg", "trips"]]
    .groupby(["veh_inc", "p_DistLrg", "a_DistLrg"], as_index=False)
    .agg(total_trips=("trips", "sum"))
)

# --------------------------------------------------------------------#
# Save Intermediate Files                                            #
# --------------------------------------------------------------------#
mod_dist_hbw_sum.to_csv("intermediate/mod_dist_hbw_sum_mc.csv", index=False)


FileNotFoundError: ``D:\GitHub\TDM-CAL-HBWDestChoice\omx_hbw_folder\pa_HBW_0veh_hi.omx`` does not exist